# Data-Driven Dispersion Initialization Analysis

Compute per-gene NB dispersion theta from method of moments on raw counts.

Variance decomposition:
- A: Poisson noise = mu_g
- B+C: Library size variability = mu_g^2 * CV^2(L)
- D: Gene-specific NB overdispersion = (mu_g^2 / theta_g) * (1 + CV^2(L))

See `.claude/plans/dispersion-init-plan.md` for full derivation.

In [ ]:
import sys

sys.path.insert(0, "../../../src")

import numpy as np
import matplotlib.pyplot as plt
from regularizedvi._dispersion_init import compute_dispersion_init

In [ ]:
# Dataset paths — all use raw counts layers
DATASETS = {
    "immune_rna": {
        "path": "../../../results/immune_integration/adata_rna.h5ad",
        "layer": "counts",
        "feature_type": None,
        "desc": "Immune integration RNA (~706k cells, 25k genes)",
    },
    "bm_rna": {
        "path": "../../../data/bmmc_multiome_multivi_neurips21_curated.h5ad",
        "layer": "counts",
        "feature_type": "GEX",
        "desc": "Bone marrow RNA (GEX, ~69k cells)",
    },
    "bm_atac": {
        "path": "../../../data/bmmc_multiome_multivi_neurips21_curated.h5ad",
        "layer": "counts",
        "feature_type": "ATAC",
        "desc": "Bone marrow ATAC (~69k cells, 116k peaks)",
    },
    "immune_atac": {
        "path": "../../../results/immune_integration/adata_atac_tiles.h5ad",
        "layer": "counts_120",
        "feature_type": None,
        "desc": "Immune integration ATAC tiles (690k x 3M) — LARGE",
    },
}

## Compute MoM theta for each dataset

In [ ]:
# Run on smaller datasets first
results = {}
for name in ["bm_rna", "bm_atac"]:
    ds = DATASETS[name]
    print(f"\n{'=' * 80}")
    print(f"Dataset: {name} — {ds['desc']}")
    print(f"{'=' * 80}")
    log_theta, diag = compute_dispersion_init(
        ds["path"],
        layer=ds["layer"],
        feature_type=ds["feature_type"],
        biological_variance_fraction=10.0,
        theta_min=0,  # no clipping for exploration
        theta_max=np.inf,
        chunk_size=5000,
    )
    results[name] = diag
    results[name]["log_theta"] = log_theta

In [ ]:
# Immune RNA (larger, ~706k cells)
name = "immune_rna"
ds = DATASETS[name]
print(f"\n{'=' * 80}")
print(f"Dataset: {name} — {ds['desc']}")
print(f"{'=' * 80}")
log_theta, diag = compute_dispersion_init(
    ds["path"],
    layer=ds["layer"],
    feature_type=ds["feature_type"],
    biological_variance_fraction=10.0,
    theta_min=0,
    theta_max=np.inf,
    chunk_size=5000,
)
results[name] = diag
results[name]["log_theta"] = log_theta

## Quantile Tables

In [ ]:
quantiles = [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
q_labels = ["min", "1%", "5%", "10%", "25%", "50%", "75%", "90%", "95%", "99%", "max"]

for name, diag in results.items():
    print(f"\n{'=' * 100}")
    print(f"Dataset: {name}")
    print(f"  CV²(L) = {diag['cv2_L']:.4f}, 1+CV² = {1 + diag['cv2_L']:.4f}")
    print(f"  Sub-Poisson genes: {diag['n_sub_poisson']}/{len(diag['mean_g'])}")
    print()

    header = f"{'':>20s} " + " ".join(f"{q:>8s}" for q in q_labels)
    print(header)
    print(f"  {'-' * 95}")

    for label, arr in [
        ("theta opt1 (full)", diag["theta_option1"]),
        ("theta opt2 (simple)", diag["theta_option2"]),
        ("log(theta opt1)", np.log(np.maximum(diag["theta_option1"], 1e-10))),
        ("mean_g", diag["mean_g"]),
        ("var_g", diag["var_g"]),
        ("excess_raw", diag["excess_raw"]),
    ]:
        vals = np.quantile(arr[np.isfinite(arr)], quantiles)
        row = f"{label:>20s} " + " ".join(f"{v:>8.3f}" for v in vals)
        print(row)

## Histograms: log(theta) distribution per dataset

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5), squeeze=False)

for i, (name, diag) in enumerate(results.items()):
    ax = axes[0, i]
    theta1 = diag["theta_option1"]
    log_t = np.log(np.clip(theta1, 1e-6, 1e8))
    finite = np.isfinite(log_t)
    ax.hist(log_t[finite], bins=100, alpha=0.7, color="steelblue", edgecolor="none")
    ax.axvline(np.log(1), color="red", ls="--", label="theta=1 (scVI default)")
    ax.axvline(np.log(9), color="orange", ls="--", label="theta=9 (our default)")
    ax.axvline(np.log(4), color="green", ls="--", label="theta=4 (disp_mean=2)")
    ax.set_title(f"{name}\nCV²(L)={diag['cv2_L']:.3f}")
    ax.set_xlabel("log(theta)")
    ax.set_ylabel("Number of genes")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## hist2d: theta vs gene mean (mean-dispersion relationship)

In [ ]:
fig, axes = plt.subplots(2, len(results), figsize=(6 * len(results), 10), squeeze=False)

for i, (name, diag) in enumerate(results.items()):
    theta1 = diag["theta_option1"]
    mean_g = diag["mean_g"]

    # Filter out extreme values for plotting
    valid = (theta1 > 1e-4) & (theta1 < 1e6) & (mean_g > 0)

    # Log-log scale
    ax = axes[0, i]
    ax.hist2d(
        np.log10(mean_g[valid]),
        np.log10(theta1[valid]),
        bins=100,
        cmap="viridis",
        cmin=1,
    )
    ax.axhline(np.log10(1), color="red", ls="--", alpha=0.7, label="theta=1")
    ax.axhline(np.log10(9), color="orange", ls="--", alpha=0.7, label="theta=9")
    ax.set_xlabel("log10(mean gene count)")
    ax.set_ylabel("log10(theta MoM)")
    ax.set_title(f"{name} — log-log")
    ax.legend(fontsize=7)

    # Linear scale (clipped)
    ax = axes[1, i]
    valid2 = valid & (theta1 < 50) & (mean_g < 50)
    if valid2.sum() > 10:
        ax.hist2d(
            mean_g[valid2],
            theta1[valid2],
            bins=100,
            cmap="viridis",
            cmin=1,
        )
    ax.axhline(1, color="red", ls="--", alpha=0.7, label="theta=1")
    ax.axhline(9, color="orange", ls="--", alpha=0.7, label="theta=9")
    ax.set_xlabel("mean gene count")
    ax.set_ylabel("theta MoM")
    ax.set_title(f"{name} — linear (clipped <50)")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## Variance decomposition: fraction of variance explained

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5), squeeze=False)

for i, (name, diag) in enumerate(results.items()):
    ax = axes[0, i]
    total = diag["var_g"]
    poisson = diag["poisson_var"]
    library = diag["library_var"]
    excess = diag["excess_raw"]

    valid = total > 0
    frac_poisson = np.median(poisson[valid] / total[valid])
    frac_library = np.median(library[valid] / total[valid])
    frac_excess = np.median(np.maximum(excess[valid], 0) / total[valid])

    ax.bar(["Poisson", "Library", "Gene-specific"], [frac_poisson, frac_library, frac_excess])
    ax.set_title(f"{name}\nMedian fraction of variance")
    ax.set_ylabel("Fraction")
    for j, v in enumerate([frac_poisson, frac_library, frac_excess]):
        ax.text(j, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Cross-dataset comparison table

In [ ]:
print(
    f"{'Dataset':<20s} {'CV²(L)':>8s} {'n_genes':>8s} {'sub-Pois':>8s} {'θ_5%':>8s} {'θ_25%':>8s} {'θ_50%':>8s} {'θ_75%':>8s} {'θ_95%':>8s}"
)
print("-" * 100)
for name, diag in results.items():
    t = diag["theta_option1"]
    finite_t = t[np.isfinite(t) & (t > 0)]
    print(
        f"{name:<20s} "
        f"{diag['cv2_L']:>8.4f} "
        f"{len(diag['mean_g']):>8d} "
        f"{diag['n_sub_poisson']:>8d} "
        f"{np.quantile(finite_t, 0.05):>8.3f} "
        f"{np.quantile(finite_t, 0.25):>8.3f} "
        f"{np.quantile(finite_t, 0.50):>8.3f} "
        f"{np.quantile(finite_t, 0.75):>8.3f} "
        f"{np.quantile(finite_t, 0.95):>8.3f}"
    )

## Comparison: Learned theta (P5) vs MoM theta

Load P5 results from extract_learned_dispersion.py and compare per-gene.

In [ ]:
import torch


def load_learned_theta(model_path):
    """Load learned theta from model checkpoint."""
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    sd = checkpoint["model_state_dict"]
    px_r_mu = sd["px_r_mu"].numpy()
    # Average across batches if gene-batch
    if px_r_mu.ndim == 2:
        px_r_mu = px_r_mu.mean(axis=1)
    return np.exp(px_r_mu)  # median theta


results_dir = "../../../results/"
p5_models = {
    "default (init=3)": f"{results_dir}immune_integration_rna_embryo_es2_no_tea_small/model/model.pt",
    "disp1 (init=1)": f"{results_dir}immune_integration_rna_embryo_es2_no_tea_small_disp1/model/model.pt",
    "disp2 (init=2)": f"{results_dir}immune_integration_rna_embryo_es2_no_tea_small_disp2/model/model.pt",
}

learned = {}
for label, path in p5_models.items():
    learned[label] = load_learned_theta(path)
    print(f"{label}: {len(learned[label])} genes, median theta = {np.median(learned[label]):.2f}")

In [ ]:
# hist2d: init=1 vs init=2 vs init=3 learned theta
pairs = [
    ("disp1 (init=1)", "default (init=3)"),
    ("disp2 (init=2)", "default (init=3)"),
    ("disp1 (init=1)", "disp2 (init=2)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (a, b) in zip(axes, pairs, strict=True):
    ta = learned[a]
    tb = learned[b]
    # Match gene counts (may differ slightly)
    n = min(len(ta), len(tb))
    ax.hist2d(
        np.log10(ta[:n]),
        np.log10(tb[:n]),
        bins=100,
        cmap="viridis",
        cmin=1,
    )
    lim = [
        min(np.log10(ta[:n]).min(), np.log10(tb[:n]).min()),
        max(np.log10(ta[:n]).max(), np.log10(tb[:n]).max()),
    ]
    ax.plot(lim, lim, "r--", alpha=0.5)
    ax.set_xlabel(f"log10(theta) — {a}")
    ax.set_ylabel(f"log10(theta) — {b}")
    ax.set_title(f"{a} vs {b}")

plt.tight_layout()
plt.show()

In [ ]:
# hist2d: MoM theta vs learned theta (immune RNA)
if "immune_rna" in results:
    mom_theta = results["immune_rna"]["theta_option1"]

    fig, axes = plt.subplots(1, len(learned), figsize=(6 * len(learned), 5))
    if len(learned) == 1:
        axes = [axes]

    for ax, (label, lt) in zip(axes, learned.items(), strict=False):
        n = min(len(mom_theta), len(lt))
        valid = (mom_theta[:n] > 1e-4) & (mom_theta[:n] < 1e6)
        ax.hist2d(
            np.log10(mom_theta[:n][valid]),
            np.log10(lt[:n][valid]),
            bins=100,
            cmap="viridis",
            cmin=1,
        )
        ax.plot([-2, 5], [-2, 5], "r--", alpha=0.5)
        ax.set_xlabel("log10(theta MoM)")
        ax.set_ylabel(f"log10(theta learned) — {label}")
        ax.set_title(f"MoM vs Learned — {label}")

    plt.tight_layout()
    plt.show()

## Summary & Decision

In [ ]:
# Print summary for decision-making
print("MoM theta summary (option 1, full correction, bio_frac=10):")
print(f"{'Dataset':<20s} {'median θ':>10s} {'mean θ':>10s} {'log(med θ)':>10s}")
for name, diag in results.items():
    t = diag["theta_option1"]
    finite_t = t[np.isfinite(t) & (t > 0) & (t < 1e8)]
    print(f"{name:<20s} {np.median(finite_t):>10.3f} {np.mean(finite_t):>10.3f} {np.log(np.median(finite_t)):>10.3f}")

print()
print("Learned theta summary (small model, dwl2=0.1):")
for label, lt in learned.items():
    print(f"  {label:<25s} median={np.median(lt):>8.3f}  log(median)={np.log(np.median(lt)):>8.3f}")